# Fig 5A — ATAC-profile UMAP colored by (RNA-defined) cell state
UMAP built from the **ATAC chromatin profile alone** (IterativeLSI on a genome-wide 500 bp TileMatrix — features independent of the RNA labels), colored by the RNA-derived `cellState`. If cells of the same state group together here, the two modalities are **concordant**. Concordance is quantified by the Adjusted Rand Index (ARI) between the unsupervised ATAC clusters and the RNA states.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
df = pd.read_csv(os.path.join(CSV_DIR, 'ATAC_UMAP.csv'))
# ATAC clusters: use the precomputed column shipped in ATAC_UMAP.csv; if absent (i.e. running
# from the full LSI on the source VM), Leiden-cluster ATAC_LSI.csv (162 MB, not in the repo).
if 'ATAC_cluster' not in df.columns:
    import scanpy as sc, anndata as ad
    lsi = pd.read_csv(os.path.join(CSV_DIR, 'ATAC_LSI.csv'), index_col=0).loc[df['cellName']]
    A = ad.AnnData(np.zeros((len(df), 1), dtype='float32')); A.obsm['X_lsi'] = lsi.values.astype('float32')
    sc.pp.neighbors(A, use_rep='X_lsi', n_neighbors=15)
    sc.tl.leiden(A, resolution=0.5, flavor='igraph', n_iterations=2, directed=False)
    df['ATAC_cluster'] = A.obs['leiden'].values
df['ATAC_cluster'] = df['ATAC_cluster'].astype(str)
mask = df['cellState'].notna() & (df['cellState'] != 'NA')
ari = adjusted_rand_score(df['cellState'][mask], df['ATAC_cluster'][mask])
nmi = normalized_mutual_info_score(df['cellState'][mask], df['ATAC_cluster'][mask])
df = df[mask]
print('ATAC cells:', len(df), '| Leiden clusters:', df['ATAC_cluster'].nunique(),
      '| ARI(state,ATACcluster)=%.2f  NMI=%.2f' % (ari, nmi))
fig, ax = plt.subplots(figsize=(3.4, 3.2))
for s in STATE_ORDER:
    m = df['cellState'] == s
    ax.scatter(df['UMAP1'][m], df['UMAP2'][m], s=1.4, c=STATE_COLORS[s],
               label=state_label(s), linewidths=0, rasterized=True)
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel('ATAC-UMAP1'); ax.set_ylabel('ATAC-UMAP2')
for sp in ['left','bottom']: ax.spines[sp].set_visible(False)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), markerscale=4,
          handletextpad=0.3, labelspacing=0.35, borderpad=0)
savepanel(fig, 'Fig5A_ATAC_UMAP_cellstate')
